In [8]:
import sys
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import dtw

# Add the probing code directory to path
sys.path.append('/mnt/polished-lake/home/annabelma/disentangling-computation-from-cot/probing/code')

# Import from load_activation_data
from load_activation_data import *
from experiment_config import * 

In [9]:
# Load activations for DTW analysis
DATA_DIR = "/mnt/polished-lake/data/mmlu_activations"

# Load activations for layer 60 (you can change this)
layer_idx = 60
activations_list, question_ids, metadata_list = load_activations_for_dtw(DATA_DIR, layer_idx=60, max_questions=2)

Found 57 categories and 1 layers
Processing category: abstract_algebra
Loaded 2 questions with activations and metadata for layer 60
Layer 60: 2 sequences, seq_len range: [2641, 8617], hidden_dim: 7168


In [10]:
# DTW Implementation using scipy cdist with cosine similarity
import torch.nn.functional as F
from scipy.spatial.distance import cdist
import numpy as np

def dtw_with_cosine_similarity_scipy(activations1, activations2):
    """
    Perform DTW between two activation sequences using cosine similarity via scipy cdist.
    
    Args:
        activations1: Tensor of shape [seq_len1, hidden_dim]
        activations2: Tensor of shape [seq_len2, hidden_dim]
    
    Returns:
        dtw_distance: The DTW distance
        path1: Indices from sequence 1
        path2: Indices from sequence 2
        distance_matrix: The cosine distance matrix used
    """
    # Convert to numpy
    seq1 = activations1.cpu().numpy()
    seq2 = activations2.cpu().numpy()
    
    # Compute cosine distance matrix using scipy cdist
    distance_matrix = cdist(seq1, seq2, metric='cosine')
    
    # DTW algorithm implementation
    len1, len2 = distance_matrix.shape
    
    # Initialize DTW matrix
    dtw_matrix = np.full((len1 + 1, len2 + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Fill DTW matrix
    for i in range(1, len1 + 1):
        for j in range(1, len2 + 1):
            cost = distance_matrix[i-1, j-1]
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],      # insertion
                dtw_matrix[i, j-1],      # deletion
                dtw_matrix[i-1, j-1]     # match
            )
    
    # Backtrack to find optimal path
    path = []
    i, j = len1, len2
    while i > 0 and j > 0:
        path.append((i-1, j-1))  # Convert back to 0-indexed
        
        # Find which direction gave minimum cost
        min_cost = min(
            dtw_matrix[i-1, j],      # insertion
            dtw_matrix[i, j-1],      # deletion
            dtw_matrix[i-1, j-1]     # match
        )
        
        if dtw_matrix[i-1, j-1] == min_cost:
            i, j = i-1, j-1
        elif dtw_matrix[i-1, j] == min_cost:
            i = i-1
        else:
            j = j-1
    
    path.reverse()
    
    # Extract path indices
    path1 = [p[0] for p in path]
    path2 = [p[1] for p in path]
    
    return dtw_matrix[len1, len2], path1, path2, distance_matrix

def cosine_distance_scipy(x, y):
    """
    Compute cosine distance between two vectors using scipy.
    
    Args:
        x: First vector
        y: Second vector
    
    Returns:
        cosine distance (1 - cosine_similarity)
    """
    # Convert to numpy if tensors
    if hasattr(x, 'cpu'):
        x = x.cpu().numpy()
    if hasattr(y, 'cpu'):
        y = y.cpu().numpy()
    
    # Ensure 2D for cdist
    x = x.reshape(1, -1)
    y = y.reshape(1, -1)
    
    # Compute cosine distance
    return cdist(x, y, metric='cosine')[0, 0]

print("DTW function with scipy cdist implemented!")


DTW function with scipy cdist implemented!


In [11]:
# DTW Visualization Functions
import matplotlib.pyplot as plt
import numpy as np

def plot_dtw_alignment(activations1, activations2, path1, path2, alignment, 
                      title="DTW Alignment", figsize=(15, 5)):
    """
    Plot DTW alignment results with multiple subplots.
    
    Args:
        activations1: First activation sequence [seq_len1, hidden_dim]
        activations2: Second activation sequence [seq_len2, hidden_dim]
        path1: DTW path indices for sequence 1
        path2: DTW path indices for sequence 2
        alignment: DTW alignment object
        title: Plot title
        figsize: Figure size
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle(title, fontsize=16)
    
    # 1. Warping path
    ax1 = axes[0, 0]
    ax1.plot(path2, path1, 'r-', linewidth=2, alpha=0.7)
    ax1.scatter(path2, path1, c='red', s=10, alpha=0.8)
    ax1.set_xlabel('Sequence 2 Position')
    ax1.set_ylabel('Sequence 1 Position')
    ax1.set_title(f'DTW Warping Path\\nDistance: {alignment.distance:.3f}')
    ax1.grid(True, alpha=0.3)
    
    # 2. Sequence lengths comparison
    ax2 = axes[0, 1]
    seq1_len, seq2_len = activations1.shape[0], activations2.shape[1]
    ax2.bar(['Sequence 1', 'Sequence 2'], [seq1_len, seq2_len], 
            color=['skyblue', 'lightcoral'], alpha=0.7)
    ax2.set_ylabel('Sequence Length')
    ax2.set_title('Sequence Lengths')
    ax2.text(0, seq1_len + max(seq1_len, seq2_len) * 0.05, f'{seq1_len}', 
             ha='center', va='bottom', fontweight='bold')
    ax2.text(1, seq2_len + max(seq1_len, seq2_len) * 0.05, f'{seq2_len}', 
             ha='center', va='bottom', fontweight='bold')
    
    # 3. Distance distribution along path
    ax3 = axes[1, 0]
    path_distances = []
    for i in range(len(path1)):
        dist = cosine_distance(activations1[path1[i]], activations2[path2[i]])
        path_distances.append(dist)
    
    ax3.plot(path_distances, 'b-', alpha=0.7)
    ax3.fill_between(range(len(path_distances)), path_distances, alpha=0.3)
    ax3.set_xlabel('Path Step')
    ax3.set_ylabel('Cosine Distance')
    ax3.set_title('Distance Along DTW Path')
    ax3.grid(True, alpha=0.3)
    
    # 4. Path statistics
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Calculate statistics
    avg_distance = np.mean(path_distances)
    max_distance = np.max(path_distances)
    min_distance = np.min(path_distances)
    
    stats_text = f'''DTW Statistics:
    
Total Distance: {alignment.distance:.3f}
Normalized Distance: {alignment.normalizedDistance:.3f}
Path Length: {len(path1)}

Distance Statistics:
Mean: {avg_distance:.3f}
Min: {min_distance:.3f}
Max: {max_distance:.3f}
Std: {np.std(path_distances):.3f}

Sequence Info:
Seq 1 Length: {seq1_len}
Seq 2 Length: {seq2_len}
Compression Ratio: {len(path1) / max(seq1_len, seq2_len):.3f}'''
    
    ax4.text(0.1, 0.9, stats_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

def plot_dtw_heatmap(activations1, activations2, path1, path2, 
                    title="DTW Distance Heatmap", figsize=(10, 8)):
    """
    Plot a heatmap of the distance matrix with DTW path overlay.
    
    Args:
        activations1: First activation sequence
        activations2: Second activation sequence  
        path1: DTW path indices for sequence 1
        path2: DTW path indices for sequence 2
        title: Plot title
        figsize: Figure size
    """
    # Create distance matrix (sample for visualization)
    seq1_len, seq2_len = activations1.shape[0], activations2.shape[1]
    
    # For large sequences, sample the distance matrix
    if seq1_len > 100 or seq2_len > 100:
        step1 = max(1, seq1_len // 100)
        step2 = max(1, seq2_len // 100)
        sample_indices1 = list(range(0, seq1_len, step1))
        sample_indices2 = list(range(0, seq2_len, step2))
        
        distance_matrix = np.zeros((len(sample_indices1), len(sample_indices2)))
        for i, idx1 in enumerate(sample_indices1):
            for j, idx2 in enumerate(sample_indices2):
                distance_matrix[i, j] = cosine_distance(activations1[idx1], activations2[idx2])
        
        # Adjust path for sampled matrix
        path1_sampled = [i // step1 for i in path1 if i % step1 == 0]
        path2_sampled = [j // step2 for j in path2 if j % step2 == 0]
        
        x_labels = [f'{i}' for i in sample_indices1[::5]]  # Show every 5th label
        y_labels = [f'{i}' for i in sample_indices2[::5]]
    else:
        distance_matrix = np.zeros((seq1_len, seq2_len))
        for i in range(seq1_len):
            for j in range(seq2_len):
                distance_matrix[i, j] = cosine_distance(activations1[i], activations2[j])
        
        path1_sampled = path1
        path2_sampled = path2
        x_labels = None
        y_labels = None
    
    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot heatmap
    im = ax.imshow(distance_matrix, cmap='viridis', aspect='auto', origin='lower')
    
    # Plot DTW path
    if len(path1_sampled) > 0 and len(path2_sampled) > 0:
        ax.plot(path2_sampled, path1_sampled, 'r-', linewidth=2, alpha=0.8)
        ax.scatter(path2_sampled, path1_sampled, c='red', s=20, alpha=0.9, zorder=5)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, label='Cosine Distance')
    
    # Labels and title
    ax.set_xlabel('Sequence 2 Position')
    ax.set_ylabel('Sequence 1 Position')
    ax.set_title(f'{title}\\nRed line shows DTW path')
    
    # Set tick labels if provided
    if x_labels is not None:
        ax.set_xticks(range(0, len(sample_indices2), 5))
        ax.set_xticklabels(x_labels)
    if y_labels is not None:
        ax.set_yticks(range(0, len(sample_indices1), 5))
        ax.set_yticklabels(y_labels)
    
    plt.tight_layout()
    plt.show()

print("DTW plotting functions implemented!")


DTW plotting functions implemented!


In [12]:
# Test DTW with scipy cdist implementation
if len(activations_list) >= 2:
    print("Running DTW with scipy cdist between first two activation sequences...")
    
    # Get the first two activation sequences
    act1 = activations_list[0]
    act2 = activations_list[1]
    
    print(f"Sequence 1 shape: {act1.shape}")
    print(f"Sequence 2 shape: {act2.shape}")
    
    # Run DTW using scipy cdist
    dtw_dist, path1, path2, distance_matrix = dtw_with_cosine_similarity_scipy(act1, act2)
    
    print(f"\nDTW Results:")
    print(f"DTW Distance: {dtw_dist:.3f}")
    print(f"DTW Path length: {len(path1)}")
    print(f"Distance matrix shape: {distance_matrix.shape}")
    print(f"Distance range: [{distance_matrix.min():.3f}, {distance_matrix.max():.3f}]")
    
    # Show some path points
    print(f"\nFirst 10 DTW path points:")
    for i in range(min(10, len(path1))):
        pos1, pos2 = path1[i], path2[i]
        # Get distance from the precomputed matrix
        dist = distance_matrix[pos1, pos2]
        print(f"  {i+1}: Position ({pos1}, {pos2}) -> Distance: {dist:.3f}")
    
    if len(path1) > 10:
        print(f"  ... and {len(path1) - 10} more points")
    
    # Compare with simple diagonal alignment
    min_len = min(act1.shape[0], act2.shape[0])
    diagonal_distance = sum(distance_matrix[i, i] for i in range(min_len))
    
    print(f"\nComparison:")
    print(f"DTW Distance: {dtw_dist:.3f}")
    print(f"Diagonal Distance: {diagonal_distance:.3f}")
    print(f"Improvement: {((diagonal_distance - dtw_dist) / diagonal_distance * 100):.1f}%")
    
    # Show metadata for the questions being compared
    if metadata_list[0] is not None and metadata_list[1] is not None:
        print(f"\nQuestion Metadata:")
        print(f"Question 1 - Model answer: {metadata_list[0].get('parsed_answer', 'N/A')}, Correct: {metadata_list[0].get('correct_answer', 'N/A')}")
        print(f"Question 2 - Model answer: {metadata_list[1].get('parsed_answer', 'N/A')}, Correct: {metadata_list[1].get('correct_answer', 'N/A')}")
    
    print("\n✅ DTW analysis with scipy cdist completed successfully!")
    
else:
    print("Need at least 2 activation sequences to run DTW")


Running DTW with scipy cdist between first two activation sequences...
Sequence 1 shape: torch.Size([2641, 7168])
Sequence 2 shape: torch.Size([8617, 7168])

DTW Results:
DTW Distance: 4285.828
DTW Path length: 8655
Distance matrix shape: (2641, 8617)
Distance range: [0.000, 1.210]

First 10 DTW path points:
  1: Position (0, 0) -> Distance: 0.000
  2: Position (1, 1) -> Distance: 0.000
  3: Position (2, 2) -> Distance: 0.000
  4: Position (3, 3) -> Distance: 0.000
  5: Position (4, 4) -> Distance: 0.000
  6: Position (5, 5) -> Distance: 0.000
  7: Position (6, 6) -> Distance: 0.000
  8: Position (7, 7) -> Distance: 0.000
  9: Position (8, 8) -> Distance: 0.000
  10: Position (9, 9) -> Distance: 0.000
  ... and 8645 more points

Comparison:
DTW Distance: 4285.828
Diagonal Distance: 1646.763
Improvement: -160.3%

Question Metadata:
Question 1 - Model answer: B, Correct: B
Question 2 - Model answer: C, Correct: C

✅ DTW analysis with scipy cdist completed successfully!


In [6]:
def dtw_with_cosine_similarity_gpu(activations1, activations2):
    """
    Perform DTW between two activation sequences using cosine similarity with GPU optimization.
    
    Args:
        activations1: Tensor of shape [seq_len1, hidden_dim] (can be on GPU)
        activations2: Tensor of shape [seq_len2, hidden_dim] (can be on GPU)
    
    Returns:
        dtw_distance: The DTW distance
        path1: Indices from sequence 1
        path2: Indices from sequence 2
        distance_matrix: The cosine distance matrix used
    """
    # Keep on GPU for cosine distance computation
    device = activations1.device
    
    # Normalize activations on GPU
    norm1 = F.normalize(activations1, p=2, dim=1)  # [seq_len1, hidden_dim]
    norm2 = F.normalize(activations2, p=2, dim=1)  # [seq_len2, hidden_dim]
    
    # Compute cosine similarity matrix on GPU
    similarity_matrix = torch.mm(norm1, norm2.t())  # [seq_len1, seq_len2]
    
    # Convert to distance matrix on GPU
    distance_matrix_gpu = 1 - similarity_matrix
    
    # Move to CPU only for DTW algorithm
    distance_matrix = distance_matrix_gpu.cpu().numpy()
    
    # DTW algorithm implementation (CPU)
    len1, len2 = distance_matrix.shape
    
    # Initialize DTW matrix
    dtw_matrix = np.full((len1 + 1, len2 + 1), np.inf)
    dtw_matrix[0, 0] = 0
    
    # Fill DTW matrix
    for i in range(1, len1 + 1):
        for j in range(1, len2 + 1):
            cost = distance_matrix[i-1, j-1]
            dtw_matrix[i, j] = cost + min(
                dtw_matrix[i-1, j],      # insertion
                dtw_matrix[i, j-1],      # deletion
                dtw_matrix[i-1, j-1]     # match
            )
    
    # Backtrack to find optimal path
    path = []
    i, j = len1, len2
    while i > 0 and j > 0:
        path.append((i-1, j-1))  # Convert back to 0-indexed
        
        # Find which direction gave minimum cost
        min_cost = min(
            dtw_matrix[i-1, j],      # insertion
            dtw_matrix[i, j-1],      # deletion
            dtw_matrix[i-1, j-1]     # match
        )
        
        if dtw_matrix[i-1, j-1] == min_cost:
            i, j = i-1, j-1
        elif dtw_matrix[i-1, j] == min_cost:
            i = i-1
        else:
            j = j-1
    
    path.reverse()
    
    # Extract path indices
    path1 = [p[0] for p in path]
    path2 = [p[1] for p in path]
    
    return dtw_matrix[len1, len2], path1, path2, distance_matrix

def cosine_distance_gpu(x, y):
    """
    Compute cosine distance between two vectors on GPU.
    
    Args:
        x: First vector (tensor)
        y: Second vector (tensor)
    
    Returns:
        cosine distance (1 - cosine_similarity)
    """
    # Normalize vectors
    x_norm = F.normalize(x.unsqueeze(0), p=2, dim=1)
    y_norm = F.normalize(y.unsqueeze(0), p=2, dim=1)
    
    # Compute cosine similarity
    similarity = torch.mm(x_norm, y_norm.t()).item()
    
    # Convert to distance
    return 1 - similarity

print("GPU-optimized DTW function implemented!")


GPU-optimized DTW function implemented!


In [ ]:
# Use the existing DTW plotting functions
if len(activations_list) >= 2 and 'dtw_dist' in locals():
    print("Generating DTW plots using existing plotting functions...")
    
    # Get the activation sequences and DTW results
    act1 = activations_list[0]
    act2 = activations_list[1]
    
    # Create alignment info object for compatibility with existing plotting functions
    class AlignmentInfo:
        def __init__(self, distance, path1, path2):
            self.distance = distance
            self.normalizedDistance = distance / len(path1)
            self.path1 = path1
            self.path2 = path2
    
    alignment_info = AlignmentInfo(dtw_dist, path1, path2)
    
    # Call the existing plotting functions
    print("1. Creating comprehensive DTW alignment plot...")
    plot_dtw_alignment(act1, act2, path1, path2, alignment_info, 
                      title=f"DTW Analysis: Layer {layer_idx}")
    
    print("2. Creating DTW distance heatmap...")
    plot_dtw_heatmap(act1, act2, path1, path2, 
                    title=f"DTW Distance Heatmap: Layer {layer_idx}")
    
    print("✅ DTW plots generated successfully!")
    
else:
    print("❌ Need to run DTW analysis first. Please run the DTW test cell above.")
